# 第一章：OpenAI入门

我这里用的是中转的qwen模型api

这是qwen官网的api文档(OPENAI API)

```python
from openai import OpenAI
import os

client = OpenAI(
    # 如果没有配置环境变量，请用阿里云百炼API Key替换：api_key="sk-xxx"
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

messages = [{"role": "user", "content": "你是谁"}]
completion = client.chat.completions.create(
    model="qwen3.5-plus",  # 您可以按需更换为其它深度思考模型
    messages=messages,
    extra_body={"enable_thinking": True},
    stream=True
)
is_answering = False  # 是否进入回复阶段
print("\n" + "=" * 20 + "思考过程" + "=" * 20)
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if hasattr(delta, "reasoning_content") and delta.reasoning_content is not None:
        if not is_answering:
            print(delta.reasoning_content, end="", flush=True)
    if hasattr(delta, "content") and delta.content:
        if not is_answering:
            print("\n" + "=" * 20 + "完整回复" + "=" * 20)
            is_answering = True
        print(delta.content, end="", flush=True)
```

In [1]:
# 一个openai示例工程
from openai import OpenAI
import os
# 加载.env文件
from dotenv import load_dotenv
load_dotenv(override=True)
QWEN_URL = os.getenv("QWEN_URL")
QWEN_API_KEY = os.getenv("QWEN_API_KEY")
QWEN_MODEL = os.getenv("QWEN_MODEL")

# 1. 直接在构造函数中设置中转站 API
client = OpenAI(
    base_url=QWEN_URL,
    api_key=QWEN_API_KEY,
)

completion = client.chat.completions.create(
    model=QWEN_MODEL,                    # 替换为您中转站支持的模型名
    messages=[
        {"role": "user", "content": "你来自哪个国家"}
    ],
    stream=True,
)

for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if hasattr(delta, "content") and delta.content:
        print(delta.content, end="", flush=True)

我是由阿里巴巴集团旗下通义实验室自主研发的大语言模型，因此可以说我来自中国。如果你有任何问题或需要帮助，随时告诉我！

# 第二章：Langchain 基础



## Memory临时会话记忆 😄

In [2]:
# 正确的导入：使用 ChatOpenAI 处理对话模型
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
import os


# 加载.env文件
from dotenv import load_dotenv
load_dotenv(override=True)
QWEN_URL = os.getenv("QWEN_URL")
QWEN_API_KEY = os.getenv("QWEN_API_KEY")
QWEN_MODEL = os.getenv("QWEN_MODEL")

def print_prompt(full_prompt):
    print("="*20, "发送给模型的提示词内容：", "="*20)
    print(full_prompt.to_string())
    print("="*50)
    return full_prompt

model = ChatOpenAI(
    base_url=QWEN_URL,
    api_key=QWEN_API_KEY,
    model=QWEN_MODEL,
)

# 使用 ChatPromptTemplate 和 MessagesPlaceholder 是处理历史记录的标准做法
prompt = ChatPromptTemplate.from_messages([
    ("system", "你需要根据对话历史来回应用户问题。"),
    MessagesPlaceholder(variable_name="my_history"),
    ("human", "{my_input}")
])

base_chain = prompt | print_prompt | model | StrOutputParser()

chat_history_store={}

def get_chat_history(user_id):
    if user_id not in chat_history_store:
        chat_history_store[user_id]=InMemoryChatMessageHistory()
    return chat_history_store[user_id]

#增强型chain

conversation_chain=RunnableWithMessageHistory(
    runnable=base_chain,
    get_session_history=get_chat_history,
    input_messages_key="my_input",
    history_messages_key="my_history",
)

if __name__ == "__main__":
    session_config = {"configurable": {"session_id": "001"}}
    # 第一次对话
    print("AI 回复:", conversation_chain.invoke({"my_input": "小明有一只狗"}, session_config))
    # 第二次对话（会自动带上第一句的记忆）
    print("AI 回复:", conversation_chain.invoke({"my_input": "小刚有两只猫"}, session_config))
    # 第三次对话（会记得小明和小刚的宠物）
    print("AI 回复:", conversation_chain.invoke({"my_input": "一共有几只动物"}, session_config))




ValidationError: 1 validation error for ChatOpenAI
model
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


## Memory长期会话记忆 😄

In [ ]:
import os
import json
from typing import Sequence, List
from langchain_core.messages import BaseMessage, messages_to_dict, messages_from_dict
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser


# 加载.env文件
from dotenv import load_dotenv
load_dotenv(override=True)
QWEN_URL = os.getenv("QWEN_URL")
QWEN_API_KEY = os.getenv("QWEN_API_KEY")
QWEN_MODEL = os.getenv("QWEN_MODEL")


# ========== 1. 修正 FileChatMessageHistory 类 ==========
class FileChatMessageHistory(BaseChatMessageHistory):
    """将聊天记录持久化到 JSON 文件"""

    def __init__(self, storage_path: str, session_id: str):
        self.storage_path = storage_path
        self.session_id = session_id
        # 文件路径 = 存储目录 / 会话ID.json
        os.makedirs(storage_path, exist_ok=True)
        self.file_path = os.path.join(storage_path, f"{session_id}.json")

    def add_message(self, message: BaseMessage) -> None:
        """添加单条消息（LangChain 要求实现此方法）"""
        # 读取现有消息
        current_messages = self.messages
        current_messages.append(message)
        # 序列化并写入文件
        serialized = messages_to_dict(current_messages)
        with open(self.file_path, "w", encoding="utf-8") as f:
            json.dump(serialized, f, ensure_ascii=False, indent=2)

    @property
    def messages(self) -> List[BaseMessage]:
        """返回当前会话的所有消息"""
        try:
            with open(self.file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                return messages_from_dict(data)
        except FileNotFoundError:
            return []

    def clear(self) -> None:
        """清空历史记录"""
        with open(self.file_path, "w", encoding="utf-8") as f:
            json.dump([], f)

# ========== 2. 配置模型和提示词 ==========
model = ChatOpenAI(
    base_url=QWEN_URL,
    api_key=QWEN_API_KEY,
    model=QWEN_MODEL,
    temperature=0.7,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "你需要根据对话历史来回应用户问题。"),
    MessagesPlaceholder(variable_name="my_history"),  # 历史记录占位符
    ("human", "{my_input}")
])

parser = StrOutputParser()
base_chain = prompt | model | parser

# ========== 3. 会话历史获取函数（修正版） ==========
def get_session_history(session_id: str) -> FileChatMessageHistory:
    """根据 session_id 返回对应的历史记录管理器"""
    return FileChatMessageHistory(
        storage_path="./chat_history",  # 固定存储目录
        session_id=session_id           # 会话ID
    )

# ========== 4. 创建带记忆的对话链 ==========
conversation_chain = RunnableWithMessageHistory(
    runnable=base_chain,
    get_session_history=get_session_history,
    input_messages_key="my_input",
    history_messages_key="my_history",
)

# ========== 5. 测试 ==========
if __name__ == "__main__":
    # 注意：config 中的 session_id 会作为参数传给 get_session_history
    session_config = {"configurable": {"session_id": "001"}}

    print("AI 回复:", conversation_chain.invoke({"my_input": "小明有一只狗"}, session_config))
    print("AI 回复:", conversation_chain.invoke({"my_input": "小刚有两只猫"}, session_config))
    print("AI 回复:", conversation_chain.invoke({"my_input": "一共有几只动物"}, session_config))

## 文档加载器

### 一、CSV加载器

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader= CSVLoader(
    file_path="./data/stu.csv",
    encoding="utf-8",
    csv_args={"delimiter":"|"}
)


#批量加载
#documents = loader.load()
#for document in documents:
#    print(type(document),document)


# 懒加载
for document in loader.lazy_load():
    print(type(document),document)

## 向量存储

### 向量内存存储

In [ ]:
from numpy.ma import ids
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import CSVLoader
from langchain_community.embeddings import DashScopeEmbeddings


import os
from dotenv import load_dotenv
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

print(f"当前获取到的 API KEY 是: {DASHSCOPE_API_KEY}")
# 缺失CSVLoader的导入语句，补上后可以解决名称未定义的错误
vectorstore = InMemoryVectorStore(
    embedding = DashScopeEmbeddings(
    dashscope_api_key=DASHSCOPE_API_KEY,
)
)




# 修复换行导致的语法问题，将参数对齐到同一行或保持语法正确的换行
loader = CSVLoader("data/info.csv", encoding="utf-8", source_column="source")

documents = loader.load()

print(documents[1])

vectorstore.add_documents(
    documents=documents,
    ids=["id"+str(i) for i in range(1,len(documents)+1)]
    )

vectorstore.delete(["id1","id2"])

results = vectorstore.similarity_search("Python是不是简单的",k=2)

print(results)


### 向量持久化存储


In [ ]:
from numpy.ma import ids
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import CSVLoader
from langchain_community.embeddings import DashScopeEmbeddings

# Chrmoa

# 加载.env文件
from dotenv import load_dotenv
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

print(f"当前获取到的 API KEY 是: {DASHSCOPE_API_KEY}")

# 缺失CSVLoader的导入语句，补上后可以解决名称未定义的错误
vectorstore = InMemoryVectorStore(
    embedding = DashScopeEmbeddings(
    dashscope_api_key=DASHSCOPE_API_KEY,
)
)

# 修复换行导致的语法问题，将参数对齐到同一行或保持语法正确的换行
loader = CSVLoader("data/info.csv", encoding="utf-8", source_column="source")

documents = loader.load()

print(documents[1])

vectorstore.add_documents(
    documents=documents,
    ids=["id"+str(i) for i in range(1,len(documents)+1)]
    )

vectorstore.delete(["id1","id2"])

results = vectorstore.similarity_search("Python是不是简单的",k=2)

print(results)
